# 🇳🇵 Nepali Corpus Collection (Model L)

Strategy: PDF/ASR first → Wiki dump (manual) → Web last

**✅ Crash-safe**: Shards write to Drive. On restart, hashes reload automatically.

**Runtime**: CPU | **Time**: ~3 hours

## 0. Setup + Resume

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/24KaratAu/nepali'
for d in ['raw/wiki_dump', 'cleaned/manual', 'cleaned/automated_nonweb',
          'cleaned/automated_web', 'cleaned/automated_culturax',
          'cleaned/automated_cc100', 'splits', 'tokenizer']:
    os.makedirs(f'{DRIVE_BASE}/{d}', exist_ok=True)
print('✅ Drive directories created')

In [ ]:
!pip install -q datasets sentencepiece huggingface_hub
print('✅ Dependencies installed')

In [ ]:
# Login to HuggingFace (needed for gated datasets like CulturaX)
# Option 1: Interactive login
from huggingface_hub import login
login()  # This will show a prompt — paste your HF token from https://huggingface.co/settings/tokens

In [ ]:
import json, hashlib, re, random, bz2, os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

LANGUAGE = 'nepali'
MIN_CHARS = 100
MIN_DEV_RATIO = 0.5
SHARD_SIZE = 10000

# Output dirs — write DIRECTLY to Drive for crash safety
NONWEB_DIR = f'{DRIVE_BASE}/cleaned/automated_nonweb'
MANUAL_DIR = f'{DRIVE_BASE}/cleaned/manual'
WEB_DIR    = f'{DRIVE_BASE}/cleaned/automated_web'
CX_DIR     = f'{DRIVE_BASE}/cleaned/automated_culturax'
CC_DIR     = f'{DRIVE_BASE}/cleaned/automated_cc100'
SPLITS_DIR = f'{DRIVE_BASE}/splits'

def devanagari_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if '\u0900' <= c <= '\u097F') / len(text)

def clean_web_text(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def text_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

# ── RESUME: reload hashes from any existing shards on Drive ──
all_hashes = set()
existing_docs = 0
existing_chars = 0
for d in [NONWEB_DIR, MANUAL_DIR, WEB_DIR, CX_DIR, CC_DIR]:
    for sf in sorted(Path(d).glob('*.jsonl')):
        with open(sf, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                doc = json.loads(line)
                h = doc.get('text_sha256', '')
                if h: all_hashes.add(h)
                existing_docs += 1
                existing_chars += doc.get('clean_text_characters', 0)

if existing_docs > 0:
    print(f'🔄 RESUMING: Found {existing_docs:,} existing docs ({existing_chars:,} chars)')
    print(f'   Loaded {len(all_hashes):,} hashes — duplicates will be auto-skipped')
else:
    print('📦 Fresh start — no existing data found')
print(f'✅ Ready | Drive: {DRIVE_BASE}')

---
## Step 1: Sangraha PDF + ASR (automated, zero wiki overlap)

**Note**: Sangraha splits are language codes (`nep`, `hin`, etc.), not `train`.

In [ ]:
from datasets import load_dataset

nonweb_docs = []
nonweb_shard_idx = len(list(Path(NONWEB_DIR).glob('*.jsonl')))
nonweb_total_chars = 0
nonweb_total_docs = 0
type_counts = Counter()

def flush_nonweb():
    global nonweb_docs, nonweb_shard_idx
    if not nonweb_docs: return
    sf = f'{NONWEB_DIR}/nonweb_shard_{nonweb_shard_idx:04d}.jsonl'
    with open(sf, 'w', encoding='utf-8') as f:
        for d in nonweb_docs:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')
    print(f'  ✅ {Path(sf).name}: {len(nonweb_docs):,} docs (saved to Drive)')
    nonweb_docs = []
    nonweb_shard_idx += 1

print('Streaming ai4bharat/sangraha verified — Nepali — PDF + ASR only...')
print('(Already-collected docs auto-skipped via hash)\n')

try:
    # Sangraha splits are language codes, not 'train'
    ds = load_dataset('ai4bharat/sangraha', 'verified', split='nep',
                      streaming=True)
    streamed = 0
    for ex in ds:
        streamed += 1
        doc_type = (ex.get('type', '') or '').strip().lower()
        type_counts[doc_type] += 1
        if 'web' in doc_type: continue  # web saved for Step 3
        
        text = ex.get('text', '')
        if not text: continue
        cleaned = clean_web_text(text)
        if len(cleaned) < MIN_CHARS: continue
        if devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
        h = text_hash(cleaned)
        if h in all_hashes: continue
        all_hashes.add(h)
        
        nonweb_docs.append({
            'doc_id': hashlib.md5(h.encode()).hexdigest()[:16],
            'source': 'sangraha', 'source_type': f'hf_sangraha_{doc_type}',
            'collection_type': 'automated', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h, 'sangraha_type': doc_type,
        })
        nonweb_total_chars += len(cleaned); nonweb_total_docs += 1
        if len(nonweb_docs) >= SHARD_SIZE: flush_nonweb()
        if nonweb_total_docs % 5000 == 0:
            print(f'  [{nonweb_total_docs:,} new non-web] {nonweb_total_chars:,} chars (streamed {streamed:,})')
    flush_nonweb()
except Exception as e:
    flush_nonweb()
    print(f'\n⚠️ Error (data saved up to this point): {e}')

print(f'\n{"="*60}')
print(f'STEP 1 COMPLETE — PDF + ASR')
print(f'  Types seen: {dict(type_counts)}')
print(f'  New non-web: {nonweb_total_docs:,} ({nonweb_total_chars:,} chars)')
print(f'  Total hashes: {len(all_hashes):,}')
print(f'{"="*60}')

---
## Step 2: Wikipedia Dump (manual, zero overlap with PDF/ASR)

In [ ]:
WIKI_DUMP = f'/content/newiki-latest-pages-articles.xml.bz2'
if not os.path.exists(WIKI_DUMP):
    print('Downloading Nepali Wikipedia dump (~50MB)...')
    !wget -q --show-progress -O {WIKI_DUMP} \
        'https://dumps.wikimedia.org/newiki/latest/newiki-latest-pages-articles.xml.bz2'
size_mb = os.path.getsize(WIKI_DUMP) / (1024*1024)
print(f'✅ Dump ready: {size_mb:.1f} MB')
!cp {WIKI_DUMP} {DRIVE_BASE}/raw/wiki_dump/ 2>/dev/null || true

In [ ]:
WIKI_MARKUP = [
    (re.compile(r'\{\{[^}]*\}\}'), ''),
    (re.compile(r'\[\[Category:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[श्रेणी:[^\]]*\]\]'), ''),
    (re.compile(r'\[\[[^|\]]*\|([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[\[([^\]]*)\]\]'), r'\1'),
    (re.compile(r'\[https?://[^\s\]]*\s*([^\]]*)\]'), r'\1'),
    (re.compile(r'\[https?://[^\]]*\]'), ''),
    (re.compile(r"'{2,5}"), ''),
    (re.compile(r'<ref[^>]*>.*?</ref>', re.DOTALL), ''),
    (re.compile(r'<ref[^/>]*/>'), ''),
    (re.compile(r'<[^>]+>'), ''),
    (re.compile(r'&[a-zA-Z]+;'), ' '),
    (re.compile(r'\[\d+\]'), ''),
]
TABLE_PAT = [
    (re.compile(r'^\s*\{\|.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|\}.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*\|-.*$', re.MULTILINE), ''),
    (re.compile(r'^\s*!.*$', re.MULTILINE), ''),
    (re.compile(r'^\|.*$', re.MULTILINE), ''),
]
SKIP_SEC = {'सन्दर्भ','सन्दर्भहरू','बाह्य कडीहरू','बाह्य लिङ्कहरू','श्रेणी',
            'यो पनि हेर्नुहोस्','स्रोत','टिप्पणी',
            'references','external links','see also','notes'}

def clean_wiki(text):
    for p,r in TABLE_PAT: text = p.sub(r, text)
    for p,r in WIKI_MARKUP: text = p.sub(r, text)
    lines, result, skip = text.split('\n'), [], False
    for line in lines:
        s = line.strip().lower()
        if s in SKIP_SEC: skip = True; continue
        if re.match(r'^={2,}.*={2,}$', s): skip = False
        if not skip: result.append(line)
    text = '\n'.join(result)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

print('✅ Wiki cleaning ready')

In [ ]:
# Auto-detect XML namespace from the dump
import io

print('Detecting XML namespace...')
with bz2.open(WIKI_DUMP, 'rt', encoding='utf-8') as f:
    # Read first 5000 chars to find namespace
    header = f.read(5000)
    ns_match = re.search(r'xmlns="(http://www\.mediawiki\.org/xml/export-[^"]+)"', header)
    if ns_match:
        NS = '{' + ns_match.group(1) + '}'
        print(f'✅ Found namespace: {NS}')
    else:
        # Try common namespaces
        NS = ''
        print('⚠️ No namespace found, trying without namespace prefix')

# Also check for mediawiki tag to confirm
print(f'Header preview: {header[:200]}...')

In [ ]:
man_docs, man_chars, man_count = [], 0, 0
man_shard = len(list(Path(MANUAL_DIR).glob('*.jsonl')))
seen, skipped_dup, rejected = 0, 0, 0

print('Processing Nepali Wikipedia dump...')
print('(PDF/ASR have zero wiki overlap → expect almost no skips)\n')

with bz2.open(WIKI_DUMP, 'rt', encoding='utf-8') as f:
    for event, elem in ET.iterparse(f, events=('end',)):
        # Handle both namespaced and non-namespaced tags
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        
        if tag != 'page': continue
        
        # Find sub-elements (namespace-aware)
        ns_el = elem.find(f'{NS}ns') if NS else elem.find('ns')
        if ns_el is None:
            elem.clear(); continue
        if ns_el.text != '0':
            elem.clear(); continue
        
        redirect = elem.find(f'{NS}redirect') if NS else elem.find('redirect')
        if redirect is not None:
            elem.clear(); continue
        
        title_el = elem.find(f'{NS}title') if NS else elem.find('title')
        id_el = elem.find(f'{NS}id') if NS else elem.find('id')
        title = title_el.text if title_el is not None else ''
        pid = id_el.text if id_el is not None else ''
        
        rev = elem.find(f'{NS}revision') if NS else elem.find('revision')
        raw = ''
        if rev is not None:
            te = rev.find(f'{NS}text') if NS else rev.find('text')
            if te is not None and te.text: raw = te.text
        elem.clear()
        if not raw.strip(): continue
        seen += 1
        
        cleaned = clean_wiki(raw)
        if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO:
            rejected += 1; continue
        h = text_hash(cleaned)
        if h in all_hashes: skipped_dup += 1; continue
        all_hashes.add(h)
        
        man_docs.append({
            'doc_id': hashlib.md5(f'{pid}_{title}'.encode()).hexdigest()[:16],
            'title': title, 'wiki_page_id': pid,
            'source': 'Nepali Wikipedia Dump', 'source_type': 'wikipedia_dump',
            'collection_type': 'manual', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h,
        })
        man_chars += len(cleaned); man_count += 1
        
        if len(man_docs) >= SHARD_SIZE:
            sf = f'{MANUAL_DIR}/wiki_shard_{man_shard:04d}.jsonl'
            with open(sf,'w',encoding='utf-8') as out:
                for d in man_docs: out.write(json.dumps(d,ensure_ascii=False)+'\n')
            print(f'  ✅ {Path(sf).name}: {len(man_docs):,} docs (saved to Drive)')
            man_docs = []; man_shard += 1
        if seen % 5000 == 0:
            print(f'  Seen:{seen:,} Accept:{man_count:,} Dup:{skipped_dup:,} Reject:{rejected:,}')

if man_docs:
    sf = f'{MANUAL_DIR}/wiki_shard_{man_shard:04d}.jsonl'
    with open(sf,'w',encoding='utf-8') as out:
        for d in man_docs: out.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'  ✅ {Path(sf).name}: {len(man_docs):,} docs (saved to Drive)')

print(f'\n{"="*60}')
print(f'STEP 2 COMPLETE — WIKI DUMP (MANUAL)')
print(f'  Articles seen: {seen:,} | Accepted: {man_count:,}')
print(f'  Skipped (dup): {skipped_dup:,} | Rejected: {rejected:,}')
print(f'  Manual chars: {man_chars:,} | Est. tokens: ~{man_chars//6:,}')
print(f'{"="*60}')

---
## Step 3: Sangraha Web + CulturaX (fill automated)

In [ ]:
web_docs, web_chars, web_count, web_skipped = [], 0, 0, 0
web_shard_idx = len(list(Path(WEB_DIR).glob('*.jsonl')))

def flush_web():
    global web_docs, web_shard_idx
    if not web_docs: return
    sf = f'{WEB_DIR}/web_shard_{web_shard_idx:04d}.jsonl'
    with open(sf,'w',encoding='utf-8') as f:
        for d in web_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'  ✅ {Path(sf).name}: {len(web_docs):,} docs (saved to Drive)')
    web_docs = []; web_shard_idx += 1

print('Streaming Sangraha Web for Nepali (split=nep)...\n')
try:
    ds = load_dataset('ai4bharat/sangraha', 'verified', split='nep',
                      streaming=True)
    for ex in ds:
        doc_type = (ex.get('type','') or '').strip().lower()
        if 'web' not in doc_type: continue
        text = ex.get('text','')
        if not text: continue
        cleaned = clean_web_text(text)
        if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
        h = text_hash(cleaned)
        if h in all_hashes: web_skipped += 1; continue
        all_hashes.add(h)
        web_docs.append({
            'doc_id': hashlib.md5(h.encode()).hexdigest()[:16],
            'source': 'sangraha', 'source_type': 'hf_sangraha_web',
            'collection_type': 'automated', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h,
        })
        web_chars += len(cleaned); web_count += 1
        if len(web_docs) >= SHARD_SIZE: flush_web()
        if web_count % 10000 == 0:
            print(f'  [{web_count:,} web] {web_chars:,} chars | overlap: {web_skipped:,}')
    flush_web()
except Exception as e:
    flush_web()
    print(f'⚠️ Error (data saved): {e}')
print(f'Sangraha Web: {web_count:,} docs, {web_chars:,} chars, {web_skipped:,} overlaps')

In [ ]:
# CulturaX (gated — requires HF login from cell above)
cx_docs, cx_chars, cx_count = [], 0, 0
cx_shard = len(list(Path(CX_DIR).glob('*.jsonl')))
print('Streaming CulturaX for Nepali...')
try:
    ds = load_dataset('uonlp/CulturaX', 'ne', split='train',
                      streaming=True)
    for ex in ds:
        text = ex.get('text','')
        if not text: continue
        cleaned = clean_web_text(text)
        if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
        h = text_hash(cleaned)
        if h in all_hashes: continue
        all_hashes.add(h)
        cx_docs.append({
            'doc_id': hashlib.md5(h.encode()).hexdigest()[:16],
            'source': 'culturax', 'source_type': 'hf_culturax',
            'collection_type': 'automated', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h,
        })
        cx_chars += len(cleaned); cx_count += 1
        if len(cx_docs) >= SHARD_SIZE:
            sf = f'{CX_DIR}/cx_shard_{cx_shard:04d}.jsonl'
            with open(sf,'w',encoding='utf-8') as f:
                for d in cx_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
            print(f'  ✅ {Path(sf).name}: {len(cx_docs):,} docs')
            cx_docs = []; cx_shard += 1
        if cx_count % 10000 == 0:
            print(f'  [{cx_count:,}] {cx_chars:,} chars')
    if cx_docs:
        sf = f'{CX_DIR}/cx_shard_{cx_shard:04d}.jsonl'
        with open(sf,'w',encoding='utf-8') as f:
            for d in cx_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
        print(f'  ✅ {Path(sf).name}: {len(cx_docs):,} docs')
except Exception as e:
    if cx_docs:
        sf = f'{CX_DIR}/cx_shard_{cx_shard:04d}.jsonl'
        with open(sf,'w',encoding='utf-8') as f:
            for d in cx_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'⚠️ CulturaX error (data saved): {e}')
print(f'CulturaX: {cx_count:,} docs, {cx_chars:,} chars')

In [ ]:
# OSCAR as fallback (instead of cc100 which is deprecated)
cc_docs, cc_chars, cc_count = [], 0, 0
cc_shard = len(list(Path(CC_DIR).glob('*.jsonl')))
print('Streaming OSCAR for Nepali...')
try:
    ds = load_dataset('oscar-corpus/OSCAR-2301', 'ne', split='train',
                      streaming=True)
    for ex in ds:
        text = ex.get('text','')
        if not text: continue
        cleaned = clean_web_text(text)
        if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
        h = text_hash(cleaned)
        if h in all_hashes: continue
        all_hashes.add(h)
        cc_docs.append({
            'doc_id': hashlib.md5(h.encode()).hexdigest()[:16],
            'source': 'oscar', 'source_type': 'hf_oscar',
            'collection_type': 'automated', 'language': LANGUAGE,
            'text': cleaned, 'clean_text_characters': len(cleaned),
            'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
            'text_sha256': h,
        })
        cc_chars += len(cleaned); cc_count += 1
        if len(cc_docs) >= SHARD_SIZE:
            sf = f'{CC_DIR}/oscar_shard_{cc_shard:04d}.jsonl'
            with open(sf,'w',encoding='utf-8') as f:
                for d in cc_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
            print(f'  ✅ {Path(sf).name}: {len(cc_docs):,} docs')
            cc_docs = []; cc_shard += 1
        if cc_count % 10000 == 0:
            print(f'  [{cc_count:,}] {cc_chars:,} chars')
    if cc_docs:
        sf = f'{CC_DIR}/oscar_shard_{cc_shard:04d}.jsonl'
        with open(sf,'w',encoding='utf-8') as f:
            for d in cc_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
        print(f'  ✅ {Path(sf).name}: {len(cc_docs):,} docs')
except Exception as e:
    if cc_docs:
        sf = f'{CC_DIR}/oscar_shard_{cc_shard:04d}.jsonl'
        with open(sf,'w',encoding='utf-8') as f:
            for d in cc_docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
    print(f'⚠️ OSCAR error (data saved): {e}')
print(f'OSCAR: {cc_count:,} docs, {cc_chars:,} chars')

In [ ]:
# Reload totals from Drive (accurate even on resume)
manual_char_total, auto_char_total = 0, 0
manual_total, auto_total = 0, 0
for d in [NONWEB_DIR, MANUAL_DIR, WEB_DIR, CX_DIR, CC_DIR]:
    for sf in Path(d).glob('*.jsonl'):
        with open(sf,'r',encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                doc = json.loads(line)
                c = doc.get('clean_text_characters', 0)
                if doc.get('collection_type') == 'manual':
                    manual_total += 1; manual_char_total += c
                else:
                    auto_total += 1; auto_char_total += c
tot = manual_char_total + auto_char_total
print(f'\n{"="*60}')
print(f'COLLECTION SUMMARY (from Drive)')
print(f'  Manual:    {manual_total:,} docs, {manual_char_total:,} chars')
print(f'  Automated: {auto_total:,} docs, {auto_char_total:,} chars')
print(f'  Total:     {manual_total+auto_total:,} docs, {tot:,} chars (~{tot//6:,} tokens)')
print(f'  Manual %:  {manual_char_total/max(1,tot)*100:.1f}%')
print(f'{"="*60}')

---
## Step 4: Split + Tokenizer + Count

In [ ]:
all_docs = []
for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR, CX_DIR, CC_DIR]:
    for sf in sorted(Path(d).glob('*.jsonl')):
        with open(sf,'r',encoding='utf-8') as f:
            for line in f:
                if line.strip(): all_docs.append(json.loads(line))
print(f'Loaded {len(all_docs):,} total docs')

random.seed(42)
groups = {}
for doc in all_docs:
    groups.setdefault(doc.get('source_type','unknown'), []).append(doc)
train, val, test = [], [], []
for src, docs in groups.items():
    random.shuffle(docs)
    n = len(docs)
    nv = max(1, int(n*0.05)) if n > 2 else 0
    nt = max(1, int(n*0.05)) if n > 2 else 0
    train.extend(docs[:n-nv-nt])
    val.extend(docs[n-nv-nt:n-nt])
    test.extend(docs[n-nt:])
random.shuffle(train); random.shuffle(val); random.shuffle(test)

for name, docs in [('train',train),('val',val),('test',test)]:
    with open(f'{SPLITS_DIR}/{name}.jsonl','w',encoding='utf-8') as f:
        for d in docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
    with open(f'{SPLITS_DIR}/{name}.txt','w',encoding='utf-8') as f:
        for d in docs:
            t = d.get('text','')
            if t.strip(): f.write(t.strip()+'\n\n')
    chars = sum(d.get('clean_text_characters',0) for d in docs)
    mn = sum(1 for d in docs if d.get('collection_type')=='manual')
    print(f'  {name:6s}: {len(docs):>8,} docs | {chars:>14,} chars | manual: {mn:,}')
print('✅ Splits saved to Drive')

In [ ]:
import sentencepiece as spm
TDIR = f'{DRIVE_BASE}/tokenizer'
MPREFIX = f'{TDIR}/nepali_bpe_32k'
TRAIN_TXT = f'{SPLITS_DIR}/train.txt'
mb = os.path.getsize(TRAIN_TXT)/(1024*1024)
print(f'Training BPE on {mb:.1f} MB...\n')

if mb < 0.01:
    print('⚠️ train.txt is empty! Cannot train tokenizer.')
    print('Please check that previous steps collected data successfully.')
else:
    spm.SentencePieceTrainer.train(
        input=TRAIN_TXT, model_prefix=MPREFIX, vocab_size=32000,
        model_type='bpe', character_coverage=0.9995, num_threads=4,
        max_sentence_length=16384, input_sentence_size=10_000_000,
        shuffle_input_sentence=True,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece='<pad>', unk_piece='<unk>', bos_piece='<bos>', eos_piece='<eos>',
        split_by_unicode_script=True, split_by_whitespace=True,
        split_digits=True, byte_fallback=True,
    )
    sp = spm.SentencePieceProcessor(); sp.load(f'{MPREFIX}.model')
    print(f'✅ Vocab: {sp.get_piece_size():,}')
    t = 'नेपाल एक दक्षिण एशियाली राष्ट्र हो।'
    print(f'Test: {t} → {sp.encode(t, out_type=str)}')

In [ ]:
if os.path.exists(f'{MPREFIX}.model'):
    sp = spm.SentencePieceProcessor(); sp.load(f'{MPREFIX}.model')
    token_counts = {}
    for sn in ['train','val','test']:
        total_tok, total_doc = 0, 0
        with open(f'{SPLITS_DIR}/{sn}.txt','r',encoding='utf-8') as f:
            buf = []
            for line in f:
                if line.strip(): buf.append(line.strip())
                elif buf:
                    total_tok += len(sp.encode(' '.join(buf))); total_doc += 1; buf = []
                    if total_doc % 50000 == 0: print(f'  {sn}: {total_doc:,} docs, {total_tok:,} tok')
            if buf: total_tok += len(sp.encode(' '.join(buf))); total_doc += 1
        token_counts[sn] = {'docs': total_doc, 'tokens': total_tok}
        print(f'  {sn}: {total_doc:,} docs → {total_tok:,} tokens')
    TOTAL = sum(v['tokens'] for v in token_counts.values())

    summary = {
        'language': LANGUAGE,
        'manual_docs': manual_total, 'manual_chars': manual_char_total,
        'automated_docs': auto_total, 'automated_chars': auto_char_total,
        'total_docs': len(all_docs), 'total_chars': tot,
        'manual_percentage': round(manual_char_total/max(1,tot)*100, 2),
        'token_counts': token_counts, 'total_tokens': TOTAL,
        'vocab_size': sp.get_piece_size(),
    }
    with open(f'{DRIVE_BASE}/corpus_summary.json','w') as f:
        json.dump(summary, f, indent=2)

    print(f'\n{"="*60}')
    print(f'🎉 NEPALI CORPUS COMPLETE!')
    print(f'  Total tokens:  {TOTAL:,}')
    print(f'  Manual %:      {manual_char_total/max(1,tot)*100:.1f}%')
    print(f'  Everything saved to: {DRIVE_BASE}')
    print(f'{"="*60}')
else:
    print('⚠️ No tokenizer model found. Please re-run after collecting data.')